In [1]:
import os
import re
import time
import random
import pickle
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from tqdm import tqdm
print('success')

success


In [2]:
# File paths
CSV_FILE     = "Publications_at_Universitas_Negeri_Surabaya_2020_-_2024.csv"
OUTPUT_FILE  = "scival_unesa_20_test.csv"

# OAuth2 login URL
LOGIN_URL = (
    "https://id.elsevier.com/as/authorization.oauth2?"
    "platSite=SVE%2FSciVal&ui_locales=en-US&scope=openid+profile+email+"
    "els_auth_info+els_analytics_info&response_type=code&"
    "redirect_uri=https%3A%2F%2Fwww.scival.com%2Fidp%2Fcode&prompt=login&"
    "client_id=SCIVAL"
)

In [13]:
load_dotenv(dotenv_path="test.env")
EMAIL    = os.getenv("UNESA_EMAIL")
PASSWORD = os.getenv("UNESA_PASSWORD")

# Cell 3: Automated login via email form + password
def create_logged_in_driver():
    opts = Options()
    opts.add_argument("--start-maximized")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=opts)

    driver.get(LOGIN_URL)
    # Dismiss cookie banner if present
    try:
        WebDriverWait(driver,5).until(
            EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
        ).click()
    except:
        pass
    driver.get(LOGIN_URL)

    # Enter email and continue
    WebDriverWait(driver,20).until(EC.element_to_be_clickable((By.ID, "bdd-email"))).send_keys(EMAIL)
    WebDriverWait(driver,10).until(EC.element_to_be_clickable((By.ID, "bdd-elsPrimaryBtn"))).click()

    # Enter password and sign in
    WebDriverWait(driver,20).until(EC.presence_of_element_located((By.ID, "bdd-password"))).send_keys(PASSWORD)
    WebDriverWait(driver,10).until(EC.element_to_be_clickable((By.ID, "bdd-elsPrimaryBtn"))).click()

    # Wait redirect to SciVal
    WebDriverWait(driver,30).until(EC.url_contains("scival.com"))
    time.sleep(3)
    return driver

driver = create_logged_in_driver()

In [14]:
# Cell 4: Load & preprocess CSV, extract first 20 URLs
with open(CSV_FILE, 'r', encoding='utf-8-sig', errors='ignore') as f:
    lines = f.readlines()
header_idx = next(i for i,line in enumerate(lines) if re.match(r'^Title,|"Title",', line))
df = pd.read_csv(CSV_FILE, skiprows=header_idx, header=0, encoding='utf-8-sig')
df.columns = [re.sub(r'\s+',' ',c).strip().lower() for c in df.columns]
link_col = next(c for c in df.columns if "scopus.com/record/display.url" in " ".join(df[c].dropna().astype(str).head(20)))
urls = df[link_col].dropna().astype(str).tolist()

In [15]:
# Cell 5: Init headless driver with loaded cookies
def extract_abstract(driver):
    try:
        WebDriverWait(driver,5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button[data-testid='Abstract']"))
        ).click()
    except:
        pass
    try:
        panel = WebDriverWait(driver,5).until(
            EC.presence_of_element_located((By.ID, "document-details-abstract"))
        )
        driver.execute_script("arguments[0].scrollIntoView(true);", panel)
        span = WebDriverWait(panel,5).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "span.Highlight_highlight__pBR3Q"))
        )
        text = span.text.strip()
        return text if len(text)>20 else None
    except:
        return None

# Cell 6: Scrape abstracts with progress bar
results = []
for url in tqdm(urls, total=len(urls), desc="Scraping abstracts"):
    driver.get(url)
    time.sleep(random.uniform(1,2))
    abstract = extract_abstract(driver)
    status = "ok" if abstract else "not_found"
    results.append({"url":url, "abstract":abstract, "status":status})

driver.quit()

# Cell 7: Merge results and save to CSV
scraped_df = pd.DataFrame(results)
df['__row_order'] = range(len(df))
merged = df.merge(scraped_df, left_on=link_col, right_on='url', how='left') \
           .sort_values('__row_order') \
           .drop(columns=['__row_order','url'])
merged.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print("✓ Scraping complete. Results saved to", OUTPUT_FILE)

Scraping abstracts: 100%|████████████████████████████████████████████████████████| 2535/2535 [2:05:18<00:00,  2.97s/it]


✓ Scraping complete. Results saved to scival_unesa_20_test.csv


In [16]:
scraped_df.head()

,url,abstract,status
0,https://www.scopus.com/record/display.url?eid=...,Purpose: This paper aims to explore employee p...,ok
1,https://www.scopus.com/record/display.url?eid=...,The focus of this research was to investigate ...,ok
2,https://www.scopus.com/record/display.url?eid=...,Critical thinking is one of the essential skil...,ok
3,https://www.scopus.com/record/display.url?eid=...,Research on Artificial Intelligence in Educati...,ok
4,https://www.scopus.com/record/display.url?eid=...,When storytelling is combined with play-based ...,ok


In [24]:
import pandas as pd
import re

def clean_abstract_final(text):
    """
    Remove copyright notices and publisher information from abstract text.
    """
    if pd.isna(text) or text is None:
        return text
    
    text = str(text).strip()
    
    # Remove everything from © symbol onwards
    text = re.sub(r'©.*$', '', text, flags=re.DOTALL)
    
    # Remove "Copyright" statements with year and organization
    text = re.sub(r'\bcopyright\s+(\d{4}|\(c\)).*$', '', text, flags=re.IGNORECASE | re.DOTALL)
    
    # Remove "All rights reserved" statements
    text = re.sub(r'\ball rights reserved.*$', '', text, flags=re.IGNORECASE | re.DOTALL)
    
    # Remove common publisher patterns at end
    publisher_patterns = [
        r'\.\s*(published by|publisher:)\s+[^.]*$',
        r'\.\s*(elsevier|springer|wiley|ieee|acm)\s+[^.]*$',
        r'\.\s*\([^)]*\d{4}[^)]*\)\s*$'  # Remove citations at end
    ]
    
    for pattern in publisher_patterns:
        text = re.sub(pattern, '.', text, flags=re.IGNORECASE | re.DOTALL)
    
    # Clean up whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\s*\.\s*$', '.', text)
    
    return text

def clean_existing_csv(input_file, output_file=None, abstract_column='abstract_y'):
    """
    Clean abstracts in existing CSV file that already contains scraped data.
    
    Parameters:
    - input_file: path to existing CSV file with scraped abstracts
    - output_file: path for cleaned CSV (if None, will overwrite input_file)
    - abstract_column: name of column containing abstracts
    """
    
    # Read existing CSV
    print(f"📖 Reading data from: {input_file}")
    df = pd.read_csv(input_file, encoding='utf-8-sig')
    
    print(f"📊 Found {len(df)} rows")
    
    # Check if abstract column exists
    if abstract_column not in df.columns:
        print(f"❌ Column '{abstract_column}' not found!")
        print(f"Available columns: {list(df.columns)}")
        return None
    
    # Count abstracts before cleaning
    abstracts_before = df[abstract_column].notna().sum()
    print(f"📝 Found {abstracts_before} non-null abstracts")
    
    # Apply cleaning function
    print("🧹 Cleaning copyright notices...")
    df[f'{abstract_column}_original'] = df[abstract_column].copy()  # Backup original
    df[abstract_column] = df[abstract_column].apply(clean_abstract_final)
    
    # Statistics
    total_chars_removed = df['characters_removed'].sum()
    cleaned_count = (df['characters_removed'] > 0).sum()
    
    print(f"✅ Cleaning complete!")
    print(f"   • {cleaned_count} abstracts were cleaned")
    print(f"   • {total_chars_removed} total characters removed")
    print(f"   • Average {total_chars_removed/max(cleaned_count,1):.1f} chars removed per cleaned abstract")
    
    # Set output file
    if output_file is None:
        output_file = input_file
    
    # Save cleaned data
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"💾 Cleaned data saved to: {output_file}")
    
    return df

INPUT_FILE = 'scival_scraped.csv' 

df_cleaned = clean_existing_csv(INPUT_FILE, abstract_column='abstract_y')


📖 Reading data from: scival_scraped.csv
📊 Found 2536 rows
📝 Found 2522 non-null abstracts
🧹 Cleaning copyright notices...
✅ Cleaning complete!
   • 2516 abstracts were cleaned
   • 128497.0 total characters removed
   • Average 51.1 chars removed per cleaned abstract
💾 Cleaned data saved to: scival_scraped.csv


In [26]:
cols_to_drop = [
    'abstract_y_original',
    'abstract_length_before',
    'abstract_length_after',
    'characters_removed',
    'has_abstract'
]

cols_existing = [col for col in cols_to_drop if col in df_cleaned.columns]

df_pruned = df_cleaned.drop(columns=cols_existing)
df_pruned.to_csv('scival_scraped.csv', index=False, encoding='utf-8-sig')
print(f"✓ Kolom {cols_existing} telah dihapus. Hasil disimpan di pruned_abstracts.csv")

✓ Kolom ['abstract_y_original', 'abstract_length_before', 'abstract_length_after', 'characters_removed', 'has_abstract'] telah dihapus. Hasil disimpan di pruned_abstracts.csv
